In [ ]:
import os
import librosa
import soundfile as sf
import numpy as np

# Function to preprocess audio files
def preprocess_wav_enhanced(src='',dst='',target_sr=16000,desired_duration=2.0,file_prefix='gunshot_',file_num=0,overlap_step=1.0):
    # Load the audio file
    data, sr = librosa.load(src, sr=target_sr)

    # Trim leading and trailing silence
    data, _ = librosa.effects.trim(data, top_db=20)

    # Normalize
    max_amp = np.max(np.abs(data))
    if max_amp > 1e-8:
        data = data / max_amp

    # Define segment length and overlap
    num_samples_2sec = int(desired_duration * sr)
    overlap_hop = int((desired_duration - overlap_step) * sr)  

    current_length = len(data)

    # If audio is shorter than the desired duration pad
    if current_length < num_samples_2sec:
        pad_length = num_samples_2sec - current_length
        data = np.pad(data, (0, pad_length), mode='constant')
        output_path = os.path.join(dst, f"{file_prefix}{file_num}.wav")
        sf.write(output_path, data, sr)
        print(f"[PADDED <2s] {src} → {output_path}")
        return

    # If audio matches the desired duration save
    if current_length == num_samples_2sec:
        output_path = os.path.join(dst, f"{file_prefix}{file_num}.wav")
        sf.write(output_path, data, sr)
        print(f"[EXACT 2s] {src} → {output_path}")
        return

    # Split audio into overlapping segments
    start = 0
    seg_count = 0

    while True:
        end = start + num_samples_2sec
        if end > current_length:
            break

        # Extract segment and save
        segment = data[start:end]
        outname = f"{file_prefix}{file_num}_{seg_count}.wav"
        output_path = os.path.join(dst, outname)
        sf.write(output_path, segment, sr)
        print(f"[SEGMENT] {src} → {output_path}")

        # Move start position
        start += overlap_hop
        seg_count += 1

    # Process remaining leftover audio
    leftover_len = current_length - start
    if leftover_len > 0:
        leftover_segment = data[start:current_length]

        # Pad leftover segment
        if leftover_len < num_samples_2sec:
            pad_amount = num_samples_2sec - leftover_len
            leftover_segment = np.pad(leftover_segment, (0, pad_amount), mode='constant')

        outname = f"{file_prefix}{file_num}_{seg_count}_leftover.wav"
        output_path = os.path.join(dst, outname)
        sf.write(output_path, leftover_segment, sr)
        print(f"[LEFTOVER PADDED] {src} → {output_path}")


In [ ]:
input_dir = r'C:\Users\jakem\Desktop\gunshot-detection-main\data\raw\not-gunshot'
output_dir = r'C:\Users\jakem\Desktop\gunshot-detection-main\NewSVMAttempt\SVMDATA\svmnot-gunshot'
file_prefix = 'not-gunshot_'

# Process all .wav files
count = 0
for file in os.listdir(input_dir):
    if file.lower().endswith('.wav'):
        file_path = os.path.join(input_dir, file)
        preprocess_wav_enhanced(
            src=file_path,
            dst=output_dir,
            target_sr=16000,
            desired_duration=2.0,
            file_prefix=file_prefix,
            file_num=count,
            overlap_step=1.0
        )
        count += 1

print(f"Processed {count} files.")


[PADDED <2s] C:\Users\jakem\Desktop\gunshot-detection-main\data\raw\not-gunshot\1-100032-A-0.wav → C:\Users\jakem\Desktop\gunshot-detection-main\NewSVMAttempt\SVMDATA\svmnot-gunshot\not-gunshot_0.wav
[SEGMENT] C:\Users\jakem\Desktop\gunshot-detection-main\data\raw\not-gunshot\1-100038-A-14.wav → C:\Users\jakem\Desktop\gunshot-detection-main\NewSVMAttempt\SVMDATA\svmnot-gunshot\not-gunshot_1_0.wav
[SEGMENT] C:\Users\jakem\Desktop\gunshot-detection-main\data\raw\not-gunshot\1-100038-A-14.wav → C:\Users\jakem\Desktop\gunshot-detection-main\NewSVMAttempt\SVMDATA\svmnot-gunshot\not-gunshot_1_1.wav
[SEGMENT] C:\Users\jakem\Desktop\gunshot-detection-main\data\raw\not-gunshot\1-100038-A-14.wav → C:\Users\jakem\Desktop\gunshot-detection-main\NewSVMAttempt\SVMDATA\svmnot-gunshot\not-gunshot_1_2.wav
[SEGMENT] C:\Users\jakem\Desktop\gunshot-detection-main\data\raw\not-gunshot\1-100038-A-14.wav → C:\Users\jakem\Desktop\gunshot-detection-main\NewSVMAttempt\SVMDATA\svmnot-gunshot\not-gunshot_1_3.wav
